# 15 — Tuned pattern viewer

All **twenty-two** chart patterns drawn at their **calibrated detector settings**, one legend toggle
each, plus standalone **trendlines** with their break markers.

Where the settings came from: a sweep on BTCUSDT 15m over the balanced nine-window cycle, 2022-2026,
re-derived by `experiments/recalibrate_tuned_specs.py` after the containment, apex and minimum-risk
gates went in — those reject shapes the earlier numbers were chosen to produce, so the old settings
had quietly become stricter than they claimed.

### Read this before you trade anything you see here

The settings were chosen to make the shapes **well-formed and frequent enough to study** — the
strictest geometry still giving at least ten detections per window in every one of the nine windows.
They were deliberately chosen **without looking at forward returns**.

**Eleven of the 22 clear that floor.** The other eleven are marked rare and set to the most permissive
combination available: pennants need a scarce symmetrical triangle *and* a preceding pole, and BTC
almost never holds a strict fixed-width sideways band for 100+ bars. They are reported as rare rather
than loosened until they appear — manufacturing detections by relaxing geometry is how a detector
starts finding shapes that are not there.

That is not excessive caution, it is the finding. Re-running the identical sweep after fixing a
lookahead bug moved the best-scoring configuration for **13 of 13 patterns** — including patterns
whose ranking across all 108 settings correlated at 0.98-0.999. The score surface is flat and its
peak is noise, so any "best settings by score" table would be a table of noise.

| what the research says | |
|---|---|
| do these patterns predict **direction**? | **No.** Hit rate **0.48** — below a coin flip, and below a shuffled null at 0.50 |
| do they predict **anything**? | **Yes — size.** 18-42% more forward range than normal, against a null at 0.97 |
| so what are they for? | A **risk signal**: expect a bigger move, size accordingly. Not a direction call |

Full evidence in [`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md).

### What each pattern draws

| drawn | meaning |
|---|---|
| thick line | the **fitted trendlines**, extended across the shape |
| dots | the **pivots** those lines are fitted through (3+ — two points always fit, so two means nothing) |
| amber line | the **boundary that had to break** for the pattern to confirm |
| dotted, coloured | the **measured-move target** |
| dotted, grey | the **protective level** |

Nothing is drawn early: a shape appears only from the bar it **confirmed**, using pivots that were
already confirmed by then. Shapes therefore show up later than you would spot them by eye — that is
what was actually knowable at the time.


## 1. Config — edit here

In [ ]:
# the kernel starts in notebooks/, so put the repo root on the path (same as the other notebooks)
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant

SYMBOL, TF = "BTCUSDT", "5m"          # the settings were calibrated on BTC 15m
START, END = "2026-05-01", "2026-05-10"
SOURCE     = "binance"                 # or "dukascopy" with SYMBOL="XAUUSD"

# None = all twenty-two. Name a subset to declutter, e.g. ["rising_wedge", "head_shoulders"].
PATTERNS = None

# which are ON when the chart opens; the rest start hidden. "sr_zones" toggles the zones.
START_VISIBLE = ("sr_zones",)

SHOW_SR_ZONES  = True
SHOW_TRENDLINES = True                 # single fitted support/resistance lines + their breaks
ZONE_LR, CLUSTER_PCT, MIN_TOUCHES = 10, 0.15, 2
EXTEND, HEIGHT = 12, 900

# standalone trendlines — see the config reference at the bottom for what each of these does
TRENDLINE = dict(atr_mult=3.5, min_touches=3, lookback_pivots=16,
                 line_tol_atr=0.6, min_span_bars=40, max_age_bars=500,
                 break_buffer_atr=0.0)

# --- added by the viewer upgrade --------------------------------------------------------
# WARM-UP: S/R zones, trendlines and a 200-EMA are only as good as the bars BEFORE them. Load
# history from WARMUP_START; the chart still opens at START. Without this the left edge shows
# indicators built from a handful of bars, which is not what they looked like at the time.
WARMUP_START = "2026-03-01"            # data starts here (feeds the indicators)
VIEW_START   = START                   # the chart OPENS here

EMAS        = (20, 50, 100, 200)       # drawn ON by default; "-ema50" in START_VISIBLE turns one off
LINE_PICKER = True                     # two dropdowns: pick a family, then isolate one instance


## 2. The tuned settings

`min_touches=3` and `line_tol_atr=0.6` are the only knobs shared by every pattern — they come from
the pooled scan, and both landed where the prior argument says they should. `atr_mult`,
`min_span_bars` and `min_height_atr` differ per pattern and each entry states its own.

`atr_mult` sizes the **zigzag**, which is a property of a whole detection pass rather than of one
pattern, so patterns calibrated to different values cannot come from a single call.
`chart_pattern_shapes_tuned` groups them by `atr_mult` and runs one pass per group — three passes
here, not thirteen.

In [ ]:
import pandas as pd
from quant.indicators import tuned_specs, TUNED_COMMON

print("shared by every pattern:", TUNED_COMMON)
pd.DataFrame(tuned_specs()).T[["atr_mult", "min_span_bars", "min_height_atr"]].sort_values(
    ["atr_mult", "min_span_bars"], ascending=[False, True])

shared by every pattern: {'min_touches_upper': 3, 'min_touches_lower': 3, 'line_tol_atr': 0.6}


## 3. Data

In [ ]:
from quant.data import get_ohlcv

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
print(f"{len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")
shown = (df.t >= pd.Timestamp(VIEW_START, tz="UTC")).sum()
print(f"{len(df)-shown:,} bars are WARM-UP; the chart opens on the {shown:,} from {VIEW_START}")

[08:17:34] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load


20,161 bars   2026-03-01 00:00:00+00:00  ->  2026-05-10 00:00:00+00:00
17,568 bars are WARM-UP; the chart opens on the 2,593 from 2026-05-01


## 4. What was found

A count before plotting. A pattern showing 0 usually means the window is too short for its
`min_span_bars` rather than anything being wrong — the wedges and the symmetrical triangle need 200
bars each, and no shape may exceed 25% of the loaded chart.

In [ ]:
from quant.indicators import chart_pattern_shapes_tuned

shapes = chart_pattern_shapes_tuned(df, patterns=PATTERNS)
if len(shapes):
    print(shapes.pattern.value_counts().to_string())
    med = shapes.span_bars.median()
    print(f"\n{len(shapes)} shapes | median width {med:.0f} bars "
          f"({100*med/len(df):.1f}% of the chart), widest {shapes.span_bars.max():.0f}")
else:
    print("nothing found — widen START/END, or drop to a faster TF")

shapes[["t", "pattern", "direction", "span_bars", "touches_upper", "touches_lower",
        "entry", "stop", "target"]].tail(12)

channel_up                34
channel_down              30
falling_wedge             25
symmetrical_triangle      17
double_top                15
rising_wedge              15
descending_triangle       15
broadening_bottom         14
ascending_triangle        13
broadening_top            10
inverse_head_shoulders     9
head_shoulders             6
channel_flat               6
rectangle_bull             6
double_bottom              5
triple_bottom              5
rectangle_bear             3
bull_flag                  3
triple_top                 2
bear_flag                  1

234 shapes | median width 157 bars (0.8% of the chart), widest 401


## 5. Chart — toggle each pattern from the legend

`tuned=True` is what applies the per-pattern settings. Without it `pattern_figure` uses one shared
configuration for everything, which is what notebook 14 does.

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    df, patterns=PATTERNS, tuned=True, start_visible=START_VISIBLE,
    sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
    cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES, extend=EXTEND,
    trendlines=SHOW_TRENDLINES, trendline_kwargs=TRENDLINE,
    emas=EMAS, view_start=VIEW_START, line_picker=LINE_PICKER,
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}  —  tuned settings", height=HEIGHT)
fig

### Open it in a browser (bigger, and it survives the notebook being closed)

In [ ]:
from quant.viz import open_in_browser

open_in_browser(fig)

---
## 6. Zoom to one pattern

Set `PICK` to a row number from section 4 to focus on that occurrence. This is the view that shows
whether a shape is real — you should see the trendlines resting on their pivots and price closing
through the amber boundary.

In [ ]:
PICK = -1        # -1 = the most recent
PAD  = 80        # bars of context each side

if len(shapes):
    row = shapes.iloc[PICK]
    lo = max(int(row.start_i) - PAD, 0)
    hi = min(int(row.confirm_i) + PAD, len(df) - 1)
    sub = df.iloc[lo:hi + 1].reset_index(drop=True)
    zoom = pattern_figure(
        sub, patterns=[row.pattern], tuned=True, start_visible=(row.pattern,),
        sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
        cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES, extend=EXTEND,
        title=f"{row.pattern}  confirmed {row.t}", height=700)
    display(zoom)
    print(f"span {row.span_bars} bars | touches {row.touches_upper}/{row.touches_lower} "
          f"| entry {row.entry:.1f} stop {row.stop:.1f} target {row.target:.1f}")

span 126 bars | touches 0/0 | entry 80666.7 stop 80111.0 target 81222.3


---
## 7. Trendlines — the single line price is respecting

A chart pattern is a closed shape made of two boundaries. A trendline is the simpler and far more
common object: **one** line under a rise or over a fall, touched repeatedly, meaningful because price
keeps turning at it — and informative when price finally closes through.

Same two rules as the shapes: the line is published only from the bar its last defining pivot
CONFIRMED, and it is FITTED by RANSAC with an envelope so nothing pokes through the side it bounds.
A line is retired once broken — the level that just failed is no longer the one being respected.

On the chart these are two extra legend toggles: dashed line, dots at the touches, and an **X at the
break**.

In [ ]:
from quant.indicators import add_trendlines, trendline_table

tl_df = add_trendlines(df, **TRENDLINE)
tl = trendline_table(tl_df)

print(f"{len(tl)} lines | {int(tl.broke.sum())} broke ({100*tl.broke.mean():.0f}%)")
print(f"breaks: {int(tl_df.tl_break_dn.sum())} down, {int(tl_df.tl_break_up.sum())} up")
print(f"a support line was live on {100*tl_df.tl_sup.notna().mean():.0f}% of bars, "
      f"resistance on {100*tl_df.tl_res.notna().mean():.0f}%")
print(f"median life {int((tl.end_i - tl.born_i).median())} bars, "
      f"median touches {tl.touches.median():.0f}")

tl[["side", "touches", "born_t", "end_t", "broke", "y_start", "y_end"]].tail(10)

206 lines | 111 broke (54%)
breaks: 54 down, 57 up
a support line was live on 16% of bars, resistance on 17%
median life 24 bars, median touches 3


### Tuning a trendline

Run this to see how each knob trades off count against quality. `min_touches` is the one that
matters most — at 2 every pair of pivots defines a line exactly, so you get many lines that mean
nothing.

In [ ]:
import pandas as pd

rows = []
for k, vals in [("atr_mult", [2.5, 3.5, 5.0]), ("min_touches", [2, 3, 4]),
                ("line_tol_atr", [0.3, 0.6, 1.2]), ("min_span_bars", [20, 40, 80]),
                ("break_buffer_atr", [0.0, 0.25, 0.5])]:
    for v in vals:
        o = add_trendlines(df, **{**TRENDLINE, k: v})
        t = trendline_table(o)
        rows.append(dict(knob=k, value=v, lines=len(t),
                         median_touches=t.touches.median() if len(t) else 0,
                         breaks=int(o.tl_break.sum()),
                         pct_bars_covered=round(100 * o.tl_sup.notna().mean(), 1)))
pd.DataFrame(rows)

---
## 8. Override one pattern

`per_pattern` takes the same shape as `tuned_specs()`, so start from it and change what you want.
Loosening `min_span_bars` is usually what you need on a short window; lowering `min_touches` below 3
is not recommended — that is the setting that produced the unusable shapes, and the pooled scan now
ranks it worst.

In [ ]:
custom = tuned_specs()
custom["symmetrical_triangle"] = {**custom["symmetrical_triangle"], "min_span_bars": 80}

alt = pattern_figure(
    df, patterns=["symmetrical_triangle"], per_pattern=custom,
    start_visible=("symmetrical_triangle",), sr_zones=False, extend=EXTEND,
    title="symmetrical triangle — min_span_bars lowered to 80", height=700)
# alt

---
## 9. Config reference — what every knob does

### Chart patterns (`PatternSpec`, per pattern via `settings={name: {...}}`)

| knob | default | what it does | raise it to |
|---|---|---|---|
| `atr_mult` | 2.5-5.0 | zigzag swing size — how big a move counts as a pivot at all. **The master dial** | get fewer, larger, more major shapes |
| `min_touches_upper` / `_lower` | 3 | pivots each boundary must pass through. **3 is the floor that means anything** — two points always fit a line exactly | demand cleaner, rarer shapes |
| `min_span_bars` | 40-200 | how wide the shape must be, in bars (absolute, not a fraction of your window) | reject short-lived noise |
| `span_cap_frac` | 0.25 | no shape may exceed this fraction of the loaded chart | — |
| `min_height_atr` | 4.0 | how tall the shape must be, in ATR | reject flat noise |
| `line_tol_atr` | 0.6 | how far off the line a pivot may sit and still count as a touch | admit sloppier fits |
| `flat_tol_atr` | 1.0 | a boundary counts as "flat" if it moves less than this many ATR across the span | call more things flat |
| `level_tol` | 0.12 | equal-highs/lows tolerance for double/triple tops and H&S shoulders, as a fraction of the shape's height | admit less symmetric tops |
| `slope_sym_max` | **2.5** | **NEW.** How unequal the two slopes may be and still count as symmetrical (steeper ÷ shallower). Applies to symmetrical triangles and broadening formations | allow lopsided "symmetrical" shapes |
| `width_tol_frac` | **0.25** | **NEW.** How much a CHANNEL's width may drift across its span. This IS the parallelism test — "fixed size with some margin of error" | allow less parallel channels |
| `converge_min_frac` | **0.25** | **NEW.** How much a triangle/wedge must actually narrow (or a broadening widen). Stops a parallel channel being called a wedge | demand more obvious convergence |

### Trendlines (`add_trendlines`)

| knob | default | what it does |
|---|---|---|
| `atr_mult` | 3.5 | zigzag swing size — the master dial for how many lines you get |
| `min_touches` | 3 | pivots the line must pass through. **3 is the floor**; at 2 every line "fits" |
| `lookback_pivots` | 16 | how many recent pivots a line may span — its memory |
| `line_tol_atr` | 0.6 | how close a pivot must sit to count as a touch |
| `min_span_bars` | 40 | minimum bars between first and last touch |
| `max_age_bars` | 500 | retire a line this old even unbroken — an ancient line is not the live one |
| `break_buffer_atr` | 0.0 | how far past the line the close must go to count as a break. Raise to filter marginal pokes |

### The three channels

`channel_up` / `channel_down` / `channel_flat` — two **parallel** boundaries holding a roughly
constant width. Direction is 0 because the tradable event is the **break**, which can go either way.
`channel_flat` is deliberately rare: BTC almost never holds a strict fixed-width sideways band for
100+ bars, and it clears no setting's sample floor. That is a property of the market, not a bug.

---
**One last framing.** Every trendline here is fitted through 3+ swing pivots by RANSAC with an
envelope constraint, so nothing pokes through the line it is supposed to bound. That is what makes
these correspond to what a person would draw.

But a well-drawn pattern is not a profitable one. On the nine-window test, `chart_short_rr8` — the
strongest pattern configuration this project found — was positive in **5 of 9** windows, exactly
matching a metronome that enters on a fixed schedule with no signal at all. Deleting the best 5
trades of ~45 takes every configuration to about **-1.0**.

Use these to see structure and to anticipate **volatility**, not direction.